# BTP: real PTB-XL Mamba validation on Kaggle

Run cells in order. This notebook has **not** been executed on a Kaggle GPU by the repository setup.
Both tasks are multi-label: **5 superclasses** and **23 official diagnostic subclasses**.
The reference Mamba architecture remains 384-wide with 8 bidirectional blocks and dropout 0.45.

| Mode | Controls | Work performed |
| --- | --- | --- |
| Safe default | All RUN/INSTALL flags False | Source/bootstrap diagnostics only; no data reads, installation or GPU work |
| Short real-data smoke | RUN_METADATA_AUDIT and RUN_SMOKE True | Both label audits, 16 training ECGs for smoke normalization, one train backward/optimizer step + one validation forward per task; **no test waveforms or benchmark metrics** |
| Full superclass | RUN_SUPERCLASS True after smoke passes | All training-fold normalization, up to FULL_EPOCHS training epochs, validation thresholds and held-out test evaluation |
| Full subclass | RUN_SUBCLASS True after smoke passes | Same complete pipeline with all 23 labels |

Keep full-run controls False until smoke passes. Smoke does not validate benchmark accuracy or full-dataset integrity.
See `docs/kaggle_execution.md` for the exact sequence and failure remedies.

## 1. Controls and source discovery (standard library only)
Attach an extracted copy of this development branch as a private Kaggle input dataset, and attach PTB-XL **v1.0.3** separately.
Set overrides if discovery finds multiple candidates. Paths must refer to extracted directories, not ZIP files.
No credentials belong in this notebook. Run All with the defaults below is safe locally.

In [ ]:
import urllib.request

try:
    urllib.request.urlopen("https://pypi.org", timeout=10)
    print("Internet is enabled ✅")
except Exception as e:
    print("Internet check failed:", e)

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

REPO_OVERRIDE = os.environ.get("BTP_REPO_DIR", "")
DATA_OVERRIDE = os.environ.get("PTBXL_DATA_DIR", "")
INSTALL_BASE = False
INSTALL_MAMBA = False
WHEEL_DIR = ""  # e.g. /kaggle/input/mamba-wheels; empty uses Internet/build tools
RUN_METADATA_AUDIT = True
RUN_SMOKE = True
RUN_SUPERCLASS = True
RUN_SUBCLASS = False
FULL_EPOCHS = 10
FULL_BATCH_SIZE = 8  # explicit memory-conscious setting; original reference batch was 64
SMOKE_BATCH_SIZE = 2  # lower to 1 on OOM; architecture remains unchanged
SMOKE_FIT_RECORDS = 16

source_candidates = []
if REPO_OVERRIDE:
    source_candidates = [Path(REPO_OVERRIDE).expanduser().resolve()]
else:
    if (Path.cwd() / "src" / "kaggle.py").is_file():
        source_candidates.append(Path.cwd().resolve())
    input_root = Path("/kaggle/input")
    if input_root.is_dir():
        source_candidates.extend(p.parent for p in input_root.rglob("requirements-kaggle.txt")
                                 if (p.parent / "src" / "kaggle.py").is_file())
source_candidates = sorted(set(source_candidates))
if len(source_candidates) != 1:
    raise RuntimeError(f"Set BTP_REPO_DIR to one source root containing src/kaggle.py and requirements-kaggle.txt. Found: {source_candidates}")
REPO_DIR = source_candidates[0]
if not (REPO_DIR / "src" / "kaggle.py").is_file() or not (REPO_DIR / "requirements-kaggle.txt").is_file():
    raise FileNotFoundError(f"Incomplete or outdated BTP source at {REPO_DIR}; attach the latest development branch")
sys.path.insert(0, str(REPO_DIR))
OUTPUT_ROOT = Path("/kaggle/working/btp_outputs") if Path("/kaggle/working").is_dir() else Path.cwd() / "outputs"
print("Source:", REPO_DIR)
print("Output root:", OUTPUT_ROOT)
print("Python:", sys.version, "Executable:", sys.executable)
print("Modes:", dict(metadata=RUN_METADATA_AUDIT, smoke=RUN_SMOKE,
                     superclass=RUN_SUPERCLASS, subclass=RUN_SUBCLASS))

## 2. Base dependency installation (opt in)
Set INSTALL_BASE=True and rerun this cell if scientific packages or wfdb are missing.
This file does not request Torch. After binary package changes, restart the kernel and rerun from cell 1 with INSTALL_BASE=False.
The log is streamed; failures retain pip's exit status and identify the command. Offline sessions need compatible wheels.

In [ ]:
def checked_install(arguments, *, env=None):
    command = [sys.executable, "-m", "pip", "install", *arguments]
    print("Running:", " ".join(command), flush=True)
    try:
        subprocess.run(command, check=True, env=env)
    except subprocess.CalledProcessError as exc:
        raise RuntimeError(f"pip failed with exit code {exc.returncode}. Read the log above; check Internet, wheel compatibility and build tools. Restart the kernel after binary changes.") from exc

if INSTALL_BASE:
    checked_install(["-r", str(REPO_DIR / "requirements-kaggle.txt")])
else:
    print("Base installation skipped. Enable INSTALL_BASE if dependency checks below fail.")

In [ ]:
%pip install wfdb==4.3.1

In [ ]:
import wfdb
print("WFDB version:", wfdb.__version__)

import torch
print("GPU:", torch.cuda.get_device_name(0))

## 3. Import and runtime diagnostics
Missing imports list the remedy. CPU checks remain possible, but CUDA is mandatory before any Mamba smoke/full run.
Torch/CUDA/Python/ABI and GPU model are recorded with smoke/full artifacts. A successful import alone does not verify CUDA kernels.

In [ ]:
import importlib
failures = {}
for package in ("numpy", "pandas", "scipy", "sklearn", "matplotlib", "torch"):
    try:
        importlib.import_module(package)
    except (ImportError, OSError, ValueError) as exc:
        failures[package] = str(exc)
if failures:
    raise RuntimeError(f"Dependency imports failed: {failures}. Run base install, preserve Kaggle CUDA Torch, restart the kernel, and rerun.")

import torch
from dataclasses import asdict
from src.config import DataConfig, ModelConfig, ExperimentConfig
from src.kaggle import find_ptbxl_root, audit_metadata, prepare_smoke_data, require_cuda, smoke_forward_backward
from src.models.registry import create_model
from src.experiments import run_experiment
from src.utils import environment_info, seed_everything, write_json
print(environment_info())
print("Torch:", torch.__version__, "CUDA runtime:", torch.version.cuda,
      "CXX11 ABI:", getattr(torch._C, "_GLIBCXX_USE_CXX11_ABI", "unavailable"))
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0), "capability:", torch.cuda.get_device_capability(0))
else:
    print("No CUDA GPU: metadata/local checks only. Select Kaggle GPU before RUN_SMOKE or full runs.")
try:
    import wfdb
    print("WFDB:", wfdb.__version__)
except ImportError:
    print("WFDB missing: enable INSTALL_BASE and restart before waveform loading.")

## 4. Mamba CUDA installation (opt in, before expensive data work)
Set INSTALL_MAMBA=True only when needed. WHEEL_DIR installs compatible attached wheels offline.
Otherwise this cell opts into the CUDA selective-scan build with MAMBA_KEEP_CUDA_BUILD=TRUE and --no-build-isolation.
Match Linux, Python, Torch, CUDA and CXX11 ABI; do not replace Kaggle's Torch with a CPU wheel.
If a core-only Mamba install already exists, install a matching CUDA wheel (or force a compatible rebuild), then restart.
[Upstream installation guidance](https://github.com/state-spaces/mamba#installation).

In [ ]:
if INSTALL_MAMBA:
    require_cuda()
    if WHEEL_DIR:
        if not Path(WHEEL_DIR).is_dir():
            raise FileNotFoundError(f"Wheel directory missing: {WHEEL_DIR}")
        checked_install(["--no-index", f"--find-links={WHEEL_DIR}", "causal-conv1d", "mamba-ssm"])
    else:
        build_env = os.environ.copy()
        build_env["MAMBA_KEEP_CUDA_BUILD"] = "TRUE"
        checked_install(["--no-build-isolation", "causal-conv1d>=1.4.0", "mamba-ssm"], env=build_env)
    print("Restart the kernel, set install flags False, rerun diagnostics and smoke.")
else:
    print("Mamba installation skipped.")

if RUN_SMOKE or RUN_SUPERCLASS or RUN_SUBCLASS:
    require_cuda()
    try:
        from mamba_ssm import Mamba
    except (ImportError, OSError, RuntimeError) as exc:
        raise RuntimeError("Mamba import failed. Inspect Torch/CUDA/ABI above, install matching CUDA packages, restart, and rerun. Original error: " + str(exc)) from exc

In [ ]:
import importlib.metadata as md
import torch

packages = ["wfdb", "causal-conv1d", "mamba-ssm"]

for package in packages:
    try:
        version = md.version(package)
        print(f"✅ {package}: {version}")
    except md.PackageNotFoundError:
        print(f"❌ {package}: NOT INSTALLED")

print("\nCUDA available:", torch.cuda.is_available())

try:
    from mamba_ssm import Mamba
    from causal_conv1d import causal_conv1d_fn
    from mamba_ssm.ops.selective_scan_interface import selective_scan_fn
    print("✅ Mamba and CUDA extension imports successful")
except Exception as e:
    print("❌ Import failed:", repr(e))

In [ ]:
import sys
import torch
import subprocess

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)

result = subprocess.run(
    ["python", "-m", "pip", "list"],
    capture_output=True,
    text=True
)

for line in result.stdout.splitlines():
    if any(x in line.lower() for x in ["wfdb", "mamba", "causal-conv"]):
        print(line)

print("Pip check completed")

In [ ]:
import os
import subprocess

print("CUDA compiler:")
subprocess.run(["which", "nvcc"])

print("\nCUDA compiler version:")
subprocess.run(["nvcc", "--version"])

print("\nAvailable disk space:")
subprocess.run(["df", "-h", "/kaggle/working"])

In [ ]:
import subprocess
import sys

for package in ["causal-conv1d", "mamba-ssm"]:
    print(f"\nChecking binary wheels for {package}...")

    result = subprocess.run(
        [
            sys.executable, "-m", "pip", "download",
            "--only-binary=:all:",
            "--no-deps",
            "--dest", "/kaggle/working/mamba_wheel_check",
            package
        ],
        capture_output=True,
        text=True,
        timeout=60
    )

    print("Available" if result.returncode == 0 else "No compatible PyPI wheel found")
    print(result.stderr[-500:] if result.returncode else result.stdout[-500:])

In [ ]:
import sys
import torch
import platform

print("Python:", sys.version.split()[0])
print("Torch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("Architecture:", platform.machine())
print("CXX11 ABI:", torch._C._GLIBCXX_USE_CXX11_ABI)

import subprocess
result = subprocess.run(
    [sys.executable, "-m", "pip", "index", "versions", "mamba-ssm"],
    capture_output=True,
    text=True
)

print("\nAvailable mamba-ssm releases:")
print(result.stdout[-1500:])
print(result.stderr[-500:])

In [ ]:
import requests

repo = "state-spaces/mamba"
url = f"https://api.github.com/repos/{repo}/releases"

response = requests.get(url, timeout=20)
response.raise_for_status()

matches = []

for release in response.json():
    for asset in release.get("assets", []):
        name = asset["name"]
        if (
            name.endswith(".whl")
            and "cp313" in name
            and "torch2.11" in name
            and "cxx11abiTRUE" in name
        ):
            matches.append(name)

if matches:
    print("Potentially compatible Mamba wheels:")
    for name in matches:
        print(name)
else:
    print("No matching Python 3.13 / PyTorch 2.11 / CXX11 ABI wheel found.")

In [ ]:
%pip install -q wfdb==4.3.1 ninja packaging

In [ ]:
import os
import sys
import subprocess

os.environ["MAMBA_KEEP_CUDA_BUILD"] = "TRUE"
os.environ["MAX_JOBS"] = "2"

log_path = "/kaggle/working/causal_conv1d_install.log"

with open(log_path, "w") as log:
    result = subprocess.run(
        [
            sys.executable, "-m", "pip", "install",
            "--no-build-isolation",
            "-v",
            "causal-conv1d>=1.4.0"
        ],
        stdout=log,
        stderr=subprocess.STDOUT
    )

print("Exit code:", result.returncode)
print("Log saved:", log_path)

with open(log_path) as f:
    print("".join(f.readlines()[-30:]))

In [ ]:
import os
import sys
import subprocess

os.environ["MAMBA_KEEP_CUDA_BUILD"] = "TRUE"
os.environ["MAX_JOBS"] = "2"

log_path = "/kaggle/working/mamba_ssm_install.log"

with open(log_path, "w") as log:
    result = subprocess.run(
        [
            sys.executable, "-m", "pip", "install",
            "--no-build-isolation",
            "-v",
            "mamba-ssm"
        ],
        stdout=log,
        stderr=subprocess.STDOUT
    )

print("Exit code:", result.returncode)
print("Log saved:", log_path)

with open(log_path) as f:
    print("".join(f.readlines()[-30:]))

In [ ]:
import torch
from mamba_ssm import Mamba

print("GPU:", torch.cuda.get_device_name(0))

model = Mamba(
    d_model=64,
    d_state=16,
    d_conv=4,
    expand=2
).cuda()

x = torch.randn(
    2, 100, 64,
    device="cuda",
    requires_grad=True
)

y = model(x)
loss = y.square().mean()
loss.backward()

print("Output shape:", y.shape)
print("Loss:", loss.item())
print("Backward pass:", "SUCCESS")
print("✅ Mamba GPU forward/backward working!")

In [ ]:
from pathlib import Path
import shutil

destination = Path("/kaggle/working/mamba_wheels")
destination.mkdir(exist_ok=True)

for package in ["causal_conv1d", "mamba_ssm"]:
    wheels = list(Path("/root/.cache/pip/wheels").rglob(f"{package}*.whl"))
    if wheels:
        wheel = max(wheels, key=lambda p: p.stat().st_mtime)
        shutil.copy2(wheel, destination / wheel.name)
        print("Saved:", wheel.name)
    else:
        print("Not found:", package)

In [ ]:
from pathlib import Path
import shutil
import sys

source = Path("/kaggle/input/datasets/arnavtiku/btp-ecg-dataset/BTP_project_files")
target = Path("/kaggle/working/BTP_project_files")

shutil.copytree(source, target, dirs_exist_ok=True)

model_file = target / "src/models/mamba.py"
code = model_file.read_text()

old = "        return self.head(pooled)"
new = "        return pooled"

assert code.count(old) == 1, "Expected exactly one matching line"
model_file.write_text(code.replace(old, new))

print("✅ Mamba encoder fixed!")

In [ ]:
sys.path.insert(0, str(target))

# Remove previously imported source modules
for module in list(sys.modules):
    if module == "src" or module.startswith("src."):
        del sys.modules[module]

print("✅ Updated source ready")

## 5. Locate the real dataset and audit both tasks
Enable RUN_METADATA_AUDIT=True first. Discovery requires both CSV files and records100/.
Zero/multiple candidates fail with instructions; a bad explicit override never silently switches datasets.
CSV schema does not certify release version: verify the attached dataset is v1.0.3. Hashes are exported.
This checks both complete vocabularies, multi-hot binary targets, folds 1–8/9/10 and pairwise patient separation **before exclusions**.
No waveform normalization/training occurs in this cell.

In [ ]:
if RUN_METADATA_AUDIT:
    DATA_DIR = find_ptbxl_root(DATA_OVERRIDE or None)
    AUDIT = audit_metadata(DATA_DIR)
    print("Dataset:", DATA_DIR)
    print(AUDIT["summary"])
    for task, table in AUDIT["statistics"].items():
        print(task, "classes:", list(AUDIT["targets"][task].columns))
        display(table)
    OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
    write_json(OUTPUT_ROOT / "metadata_audit.json", AUDIT["summary"])
    for task, table in AUDIT["statistics"].items():
        table.to_csv(OUTPUT_ROOT / f"{task}_class_statistics.csv", index=False)
else:
    print("Metadata audit skipped. Enable RUN_METADATA_AUDIT before smoke/full runs.")

## 6. Freeze experiment settings
Keep 100 Hz, 10 s and channels_first for this milestone. Smoke uses the same filters and architecture,
but no augmentation and only a tiny training-only normalization fit. Full runs fit **all training records** separately.
Both full-run cells honor FULL_EPOCHS and FULL_BATCH_SIZE here; no hardcoded epoch override.
Batch 8 is a declared memory adjustment; original reference batch was 64. GPU memory feasibility remains to be measured.

In [ ]:
CONFIG = ExperimentConfig(
    epochs=FULL_EPOCHS, batch_size=FULL_BATCH_SIZE, device="cuda",
    data=DataConfig(data_dir=str(DATA_DIR) if RUN_METADATA_AUDIT else DATA_OVERRIDE,
                    sampling_rate=100, duration=10, num_workers=2, augment=True),
    model=ModelConfig(), output_root=str(OUTPUT_ROOT),
)
print(asdict(CONFIG))

## 7. Short smoke: load real ECGs and initialize preserved Mamba
Enable RUN_SMOKE=True with RUN_METADATA_AUDIT=True. Each task reads at most SMOKE_FIT_RECORDS
training waveforms plus one tiny training/validation batch; **no fold-10 waveform is read**.
WFDB verifies 100 Hz; the smoke reader requires the original (12,1000) shape rather than silently padding it.
Both 5/23 heads use the full preserved encoder. On OOM reduce SMOKE_BATCH_SIZE to 1 and restart;
do not silently shrink the reference architecture. The next cell performs backward and a validation forward.

In [ ]:
SMOKE_REPORTS = {}
if RUN_SMOKE:
    if not RUN_METADATA_AUDIT or "AUDIT" not in globals():
        raise RuntimeError("Enable RUN_METADATA_AUDIT and run the audit cell first")
    require_cuda()
    SMOKE_DIR = OUTPUT_ROOT / "smoke"
    SMOKE_DIR.mkdir(parents=True, exist_ok=True)
    # Invalidate old success before attempting a fresh smoke run.
    write_json(SMOKE_DIR / "status.json", {"status": "started", "benchmark_result": False})
    SMOKE_BATCHES = {}
    for task in ("superclass", "subclass"):
        SMOKE_BATCHES[task] = prepare_smoke_data(AUDIT, task,
            fit_records=SMOKE_FIT_RECORDS, batch_size=SMOKE_BATCH_SIZE, config=CONFIG.data)
        x, y, ids = SMOKE_BATCHES[task]["batches"]["train"]
        print(task, "real train shapes:", tuple(x.shape), tuple(y.shape), "IDs:", ids.tolist())
else:
    print("Real waveform smoke preparation skipped.")

In [ ]:
if RUN_SMOKE:
    import gc
    write_json(SMOKE_DIR / "environment.json", environment_info())
    write_json(SMOKE_DIR / "config.json", asdict(CONFIG))
    for task in ("superclass", "subclass"):
        seed_everything(CONFIG.seed)
        smoke_model = None
        try:
            smoke_model = create_model("mamba", len(SMOKE_BATCHES[task]["class_names"]), CONFIG.model)
            report = smoke_forward_backward(smoke_model, SMOKE_BATCHES[task],
                                              device="cuda", mixed_precision=CONFIG.mixed_precision)
            report["model_config"] = asdict(CONFIG.model)
            report["data_config"] = asdict(CONFIG.data)
            report["metadata_sha256"] = AUDIT["summary"]["metadata_sha256"]
            write_json(SMOKE_DIR / f"{task}_smoke.json", report)
            SMOKE_REPORTS[task] = report
            print("SMOKE PASSED (not benchmark training):", task, report)
        except Exception as exc:
            write_json(SMOKE_DIR / "status.json", {"status": "failed", "task": task, "error": str(exc)})
            raise RuntimeError(f"{task} smoke failed: {exc}. Check waveform path/fs, CUDA wheels and ABI. For OOM use batch 1 and restart. No full-training success is implied.") from exc
        finally:
            del smoke_model
            gc.collect()
            torch.cuda.empty_cache()
    write_json(SMOKE_DIR / "status.json", {"status": "passed", "tasks": list(SMOKE_REPORTS), "benchmark_result": False})
else:
    print("Forward/backward smoke skipped; no GPU validation claimed.")

## 8. Full superclass training/evaluation (explicit opt in)
Set RUN_SUPERCLASS=True only after the current-session superclass smoke passes.
This starts the entire training pipeline: complete training-fold normalization, full train/validation loaders,
early stopping, best checkpoint, validation-only thresholds, then **held-out fold 10** evaluation.
A single epoch on full data is still a full-dataset run; it is not the short smoke test.
Full models are initialized fresh: smoke-updated weights are discarded.

In [ ]:

from pathlib import Path
import importlib
import sys

# Writable project folder
project_dir = Path("/kaggle/working/BTP_project_files")
data_file = project_dir / "src" / "data.py"

assert data_file.exists(), f"File not found: {data_file}"

code = data_file.read_text()

# Existing normalization code
old = '''    normalizer.fit(fit_ds.processed_signal(i) for i in original_splits["train"])'''

# Faster normalization using 1000 training ECGs
new = '''    # Fast, reproducible normalization using training ECGs only
    import numpy as np

    train_ids = np.array(original_splits["train"])
    rng = np.random.default_rng(seed)

    sample_size = min(1000, len(train_ids))
    sampled_ids = rng.choice(
        train_ids,
        size=sample_size,
        replace=False
    )

    print(
        f"Fitting normalization using {sample_size} ECGs...",
        flush=True
    )

    normalizer.fit(
        fit_ds.processed_signal(int(i))
        for i in sampled_ids
    )

    print("Normalization complete!", flush=True)'''

if old in code:
    data_file.write_text(code.replace(old, new, 1))
    print("✅ Normalization optimization applied.")
elif "Fitting normalization using {sample_size} ECGs" in code:
    print("✅ Optimization already applied.")
else:
    raise RuntimeError(
        "Expected normalization code not found. "
        "No changes were made."
    )

# Ensure Python uses the writable project
sys.path.insert(0, str(project_dir))

# Reload updated data and experiment modules
import src.data
import src.experiments

importlib.reload(src.data)
importlib.reload(src.experiments)

# Update run_experiment in the current notebook
run_experiment = src.experiments.run_experiment

print("✅ Updated training pipeline loaded.")
print("Source:", src.data.__file__)


In [ ]:
def require_smoke_pass(task):
    if not RUN_METADATA_AUDIT or "AUDIT" not in globals():
        raise RuntimeError("Run the metadata audit first")
    report = SMOKE_REPORTS.get(task)
    if report is None or report["device"] != "cuda":
        raise RuntimeError(f"Run the real-data CUDA forward/backward smoke for {task} in this session first")
    if (report["model_config"] != asdict(CONFIG.model)
            or report["data_config"] != asdict(CONFIG.data)
            or report["metadata_sha256"] != AUDIT["summary"]["metadata_sha256"]):
        raise RuntimeError("Configuration/data changed since smoke; rerun smoke with the new settings")
    require_cuda()

if RUN_SUPERCLASS:
    require_smoke_pass("superclass")
    superclass_result = run_experiment(model_name="mamba", task="superclass", config=CONFIG)
    print("Completed full superclass experiment:", superclass_result["output_dir"])
    display(superclass_result["history"])
    display(superclass_result["summary"])
else:
    print("Full superclass run skipped.")

## 9. Full subclass training/evaluation (independent opt in)
Set RUN_SUBCLASS=True after its own smoke passes. Uses all **23 official subclasses** without dropping rare columns.
All signals, folds, preprocessing and metrics match the superclass pipeline. Do not tune using test results.

In [ ]:
if RUN_SUBCLASS:
    require_smoke_pass("subclass")
    subclass_result = run_experiment(model_name="mamba", task="subclass", config=CONFIG)
    print("Completed full subclass experiment:", subclass_result["output_dir"])
    display(subclass_result["history"])
    display(subclass_result["summary"])
else:
    print("Full subclass run skipped.")

## 10. Verify exports and display curves
Only completed full runs are checked here. Each unique run folder must contain the artifacts below.
Missing files fail instead of implying completion. Save `/kaggle/working/btp_outputs` as Kaggle notebook output.
`smoke/` contains smoke reports only; its losses are not benchmark metrics.
Missing ranking metrics in rare classes are JSON null, with valid class counts reported.

In [ ]:
REQUIRED_EXPORTS = ["config.json", "environment.json", "data_manifest.json", "class_statistics.csv",
    "best.pth", "last.pth", "history.csv", "thresholds.json", "summary.json",
    "figures/loss_curves.png", "figures/confusion_matrices.png"]
REQUIRED_EXPORTS += [f"{split}/{name}" for split in ("validation", "test", "test_at_0.5")
                     for name in ("metrics.json", "per_class.csv", "confusion_matrices.csv", "predictions.csv")]
for task, enabled in (("superclass", RUN_SUPERCLASS), ("subclass", RUN_SUBCLASS)):
    if not enabled:
        continue
    result = globals()[f"{task}_result"]
    directory = Path(result["output_dir"])
    missing = [name for name in REQUIRED_EXPORTS if not (directory / name).is_file()]
    if missing:
        raise RuntimeError(f"Incomplete {task} exports at {directory}: {missing}")
    print(task, "verified exports:", directory)
    from IPython.display import Image, display
    display(Image(filename=str(directory / "figures/loss_curves.png")))

## Resume and evidence to bring back
Resume with the original unchanged config, run_dir and resume_from=<run_dir>/last.pth,
keeping best.pth. See README. Exact CPU continuation is tested with zero workers; CUDA kernels and
multi-worker augmentation need not be bitwise reproducible. Historical checkpoints use different configurations.

Return the smoke status/reports, environment/config, dataset metadata hashes and any complete full-run
output folders. If failure occurs, return the full failing-cell traceback and pip log/runtime diagnostics.
No success is claimed for stages whose cells were skipped or failed. No extra architectures or SSL are run.